**Import de modules :**

In [10]:
import re
from pyspark.sql.functions import (
    col, when, lit, isnan,
    avg, count, min, max, corr,
    sum as spark_sum,
    round as spark_round,
    abs as spark_abs
)
from pyspark.sql.types import FloatType, IntegerType

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 12, Finished, Available, Finished, False)

**Chargement depuis Bronze :**

In [11]:
df_bronze = spark.read.format("delta").table("bronze_supply_chain_raw")
print(f"Bronze : {df_bronze.count()} lignes")
display(df_bronze.limit(3))

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 13, Finished, Available, Finished, False)

Bronze : 100 lignes


SynapseWidget(Synapse.DataFrame, d0ab0c42-7c2d-44e0-971b-b2d1dbc27831)

**Renommage des colonnes en snake_case :**
Les colonnes du dataset ont des majuscules; on les normalise

In [12]:
def to_snake_case(name):
    name = re.sub(r'[\s\-/]+', '_', name)
    name = re.sub(r'[^a-zA-Z0-9_]', '', name)
    return name.lower().strip('_')

renamed = {c: to_snake_case(c) for c in df_bronze.columns}
df = df_bronze
for old, new in renamed.items():
    if old != new:
        df = df.withColumnRenamed(old, new)

print("Colonnes après renommage :")
print(df.columns)

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 14, Finished, Available, Finished, False)

Colonnes après renommage :
['product_type', 'sku', 'price', 'availability', 'number_of_products_sold', 'revenue_generated', 'customer_demographics', 'stock_levels', 'lead_times', 'order_quantities', 'shipping_times', 'shipping_carriers', 'shipping_costs', 'supplier_name', 'location', 'lead_time', 'production_volumes', 'manufacturing_lead_time', 'manufacturing_costs', 'inspection_results', 'defect_rates', 'transportation_modes', 'routes', 'costs']


**Typage correct des colonnes :**

In [13]:
numeric_cols = [
    "price", "revenue_generated",
    "shipping_costs", "manufacturing_costs",
    "defect_rates", "costs"
]

for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(FloatType()))

int_cols = ["availability", "number_of_products_sold", 
            "stock_levels", "lead_times", "order_quantities",
            "shipping_times", "lead_time","production_volumes",
            "manufacturing_lead_time"
]
for c in int_cols:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(IntegerType()))

df.printSchema()

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 15, Finished, Available, Finished, False)

root
 |-- product_type: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- price: float (nullable = true)
 |-- availability: integer (nullable = true)
 |-- number_of_products_sold: integer (nullable = true)
 |-- revenue_generated: float (nullable = true)
 |-- customer_demographics: string (nullable = true)
 |-- stock_levels: integer (nullable = true)
 |-- lead_times: integer (nullable = true)
 |-- order_quantities: integer (nullable = true)
 |-- shipping_times: integer (nullable = true)
 |-- shipping_carriers: string (nullable = true)
 |-- shipping_costs: float (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- production_volumes: integer (nullable = true)
 |-- manufacturing_lead_time: integer (nullable = true)
 |-- manufacturing_costs: float (nullable = true)
 |-- inspection_results: string (nullable = true)
 |-- defect_rates: float (nullable = true)
 |-- transportation_mod

**Suppression des doublons et gestion des nulls :**

In [14]:
# Doublons
before = df.count()
df = df.dropDuplicates()
after = df.count()
print(f"Doublons supprimés : {before - after}")

# Nulls sur colonnes critiques
critical_cols = ["sku", "product_type", "revenue_generated"]
df = df.dropna(subset=critical_cols)

# Nulls numériques → 0
fill_zero = {c: 0.0 for c in numeric_cols if c in df.columns}
df = df.fillna(fill_zero)

print(f"Lignes finales Silver : {df.count()}")

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 16, Finished, Available, Finished, False)

Doublons supprimés : 0
Lignes finales Silver : 100


**Résolution de l'ambiguïté lead_time / lead_times :** Deux colonnes prêtent à confusion dans notre jeu de données lead_time et lead_times. Nous allons essayer de comprendre à quoi réfère chacune de ces colonnes

In [15]:
# ============================================================
# DÉCISION DATA QUALITY — lead_time vs lead_times
# ============================================================
# Source : documentation Kaggle + clarification discussion communauté
#
#   - lead_time  (Lead Time)   → INBOUND lead time
#                                 Délai fournisseur → entreprise
#                                 "time to receive raw materials
#                                  from vendors/suppliers"
#
#   - lead_times (Lead Times)  → OUTBOUND lead time
#                                 Délai entreprise → client
#                                 "time to ship products
#                                  to customers"
#
#   - manufacturing_lead_time  → Délai de fabrication interne
#                                 "time to produce a product"
#
# Ces trois colonnes couvrent le cycle supply chain complet :
#   Fournisseur → [inbound] → Usine → [manufacturing] → Entrepôt → [outbound] → Client
#
# KPI dérivé clé : end_to_end_lead_time = inbound + manufacturing + outbound
# ============================================================

df = df \
    .withColumnRenamed("lead_time",  "inbound_lead_time") \
    .withColumnRenamed("lead_times", "outbound_lead_time")

print("Renommage effectué.")
display(df.select(
    "sku", "supplier_name",
    "inbound_lead_time",
    "manufacturing_lead_time",
    "outbound_lead_time"
).limit(10))

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 17, Finished, Available, Finished, False)

Renommage effectué.


SynapseWidget(Synapse.DataFrame, 05e5cf93-111b-4581-8dc8-0feaf92e835a)

**Colonnes dérivées métier (KPI's) :**

In [ ]:
# KPI 1 : Marge bénéficiaire (%)
df = df.withColumn(
    "profit_margin_pct",
    spark_round(
        (col("revenue_generated") - col("manufacturing_costs") - col("shipping_costs"))
        / col("revenue_generated") * 100, 2
    )
)

# KPI 2 : End-to-end lead time (cycle supply chain complet)
df = df.withColumn(
    "end_to_end_lead_time",
    col("inbound_lead_time") + col("manufacturing_lead_time") + col("outbound_lead_time")
)

# KPI 3 : Part de chaque segment dans le lead time total (%)
df = df \
    .withColumn(
        "inbound_lead_time_pct",
        spark_round(col("inbound_lead_time") / col("end_to_end_lead_time") * 100, 1)
    ) \
    .withColumn(
        "manufacturing_lead_time_pct",
        spark_round(col("manufacturing_lead_time") / col("end_to_end_lead_time") * 100, 1)
    ) \
    .withColumn(
        "outbound_lead_time_pct",
        spark_round(col("outbound_lead_time") / col("end_to_end_lead_time") * 100, 1)
    )

# KPI 4 : Segment goulot d'étranglement par SKU
df = df.withColumn(
    "bottleneck_segment",
    when(
        (col("inbound_lead_time") >= col("manufacturing_lead_time")) &
        (col("inbound_lead_time") >= col("outbound_lead_time")),
        lit("inbound")
    ).when(
        col("manufacturing_lead_time") >= col("outbound_lead_time"),
        lit("manufacturing")
    ).otherwise(lit("outbound"))
)

# KPI 5 : Flag criticité end-to-end
df = df.withColumn(
    "e2e_delay_flag",
    when(col("end_to_end_lead_time") > 60, lit("critical"))
    .when(col("end_to_end_lead_time") > 40, lit("at_risk"))
    .otherwise(lit("on_time"))
)

# KPI 6 : Couverture de stock
df = df.withColumn(
    "stock_coverage",
    when(
        col("order_quantities") > 0,
        spark_round(col("stock_levels") / col("order_quantities"), 2)
    ).otherwise(lit(0.0))
)

# KPI 7 : Flag qualité défauts
df = df.withColumn(
    "quality_flag",
    when(col("defect_rates") > 3.5, lit("high_defect"))
    .when(col("defect_rates") > 1.5, lit("medium_defect"))
    .otherwise(lit("ok"))
)

print("Colonnes dérivées créées avec succès.")
display(df.select(
    "sku", "product_type",
    "profit_margin_pct",
    "inbound_lead_time", "manufacturing_lead_time", "outbound_lead_time",
    "end_to_end_lead_time", "bottleneck_segment", "e2e_delay_flag",
    "stock_coverage", "quality_flag"
).limit(10))

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 18, Finished, Available, Finished, False)

Colonnes dérivées créées avec succès.


SynapseWidget(Synapse.DataFrame, 14737289-65c4-48ca-9f5a-7e0260a2ec82)

**Contrôle qualité avant sauvegarde :**

In [17]:
print("=== Contrôle qualité Silver ===\n")

print(f"Dimensions finales : {df.count()} lignes x {len(df.columns)} colonnes\n")

print("--- Nulls sur toutes les colonnes ---")
display(
    df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
)

print("\n--- KPIs de synthèse par type de produit ---")
display(
    df.groupBy("product_type").agg(
        count("sku").alias("nb_skus"),
        spark_round(avg("revenue_generated"), 2).alias("avg_revenue"),
        spark_round(avg("profit_margin_pct"), 2).alias("avg_margin_pct"),
        spark_round(avg("end_to_end_lead_time"), 1).alias("avg_e2e_lead_time"),
        spark_round(avg("defect_rates"), 4).alias("avg_defect_rate")
    ).orderBy("avg_revenue", ascending=False)
)

print("\n--- Distribution des goulots d'étranglement ---")
display(
    df.groupBy("bottleneck_segment").count()
      .orderBy("count", ascending=False)
)

print("\n--- Distribution des flags e2e ---")
display(
    df.groupBy("e2e_delay_flag").count()
      .orderBy("count", ascending=False)
)

print("\n--- Distribution des flags qualité ---")
display(
    df.groupBy("quality_flag").count()
      .orderBy("count", ascending=False)
)

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 19, Finished, Available, Finished, False)

=== Contrôle qualité Silver ===

Dimensions finales : 100 lignes x 33 colonnes

--- Nulls sur toutes les colonnes ---


SynapseWidget(Synapse.DataFrame, bb99fa8e-daf8-48f4-92bb-2d30845e00c2)


--- KPIs de synthèse par type de produit ---


SynapseWidget(Synapse.DataFrame, 3e0c49ae-033e-4cc5-9a6f-72677bd3e92e)

...

SynapseWidget(Synapse.DataFrame, 5eb25870-5b4e-44bf-9801-231b186320a0)


--- Distribution des flags e2e ---


SynapseWidget(Synapse.DataFrame, a0245253-0780-487b-8e4d-f6dd575f484e)


--- Distribution des flags qualité ---


SynapseWidget(Synapse.DataFrame, 0c79be7b-18f2-49b2-9fd9-c5bd77006713)

**Sauvegarde en Delta Table Silver :**

In [18]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_supply_chain")

print("Table silver_supply_chain sauvegardée avec succès.")
print(f"Dimensions finales : {df.count()} lignes x {len(df.columns)} colonnes")

# Vérification lecture
df_check = spark.read.format("delta").table("silver_supply_chain")
print(f"Vérification lecture : {df_check.count()} lignes")

StatementMeta(, 5f14ff84-d921-4aff-be84-22e898eeba84, 20, Finished, Available, Finished, False)

Table silver_supply_chain sauvegardée avec succès.
Dimensions finales : 100 lignes x 33 colonnes
Vérification lecture : 100 lignes
